# 🚀 Deep Social Sentiment Analysis — Colab Training

**Chạy theo thứ tự từ trên xuống. Mỗi section có hướng dẫn rõ ràng.**

| Section | Việc làm | Thời gian |
|---|---|---|
| 0 | Kiểm tra GPU | < 1 phút |
| 1 | Clone repo từ GitHub | < 1 phút |
| 2 | Mount Google Drive + link data | 1–2 phút |
| 3 | Cài thư viện | 3–5 phút |
| 4 | Pseudo-label 990 Facebook posts | 10–20 phút |
| 5 | Chuẩn bị dataset (merge + split) | 1–2 phút |
| 6 | Training model | 1–4 giờ (tuỳ GPU) |
| 7 | Ablation study | 3–8 giờ |
| 8 | Evaluate + save kết quả | < 5 phút |

## 0. Kiểm tra GPU

Vào **Runtime → Change runtime type → T4 GPU** trước khi chạy.

In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB VRAM)')
    DEVICE = 'cuda'
else:
    print('⚠️  GPU không khả dụng — vào Runtime → Change runtime type → T4 GPU')
    DEVICE = 'cpu'

!nvidia-smi 2>/dev/null | head -15 || echo '(nvidia-smi not available)'

## 1. Clone repo từ GitHub

Repo là **public** → không cần token. Nếu bạn đổi sang private, xem cell bên dưới.

In [ ]:
import os

GITHUB_USER = 'nhiney'                          # ← username GitHub của bạn
REPO_NAME   = 'deep-social-sentiment-analysis'  # ← tên repo
REPO_URL    = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR    = f'/content/{REPO_NAME}'

# Clone (hoặc pull nếu đã tồn tại)
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f'✅ Cloned → {REPO_DIR}')
else:
    !cd {REPO_DIR} && git pull origin main
    print(f'✅ Pulled latest → {REPO_DIR}')

# Đặt working directory = project root
os.chdir(REPO_DIR)
!pwd
!ls -la

In [ ]:
# ── CHỈ chạy cell này nếu repo là PRIVATE ──────────────────────────────────
# Bỏ qua nếu repo public.
#
# Cách lấy token:
#   GitHub.com → Settings → Developer settings
#   → Personal access tokens → Tokens (classic)
#   → Generate new token → chọn 'repo' scope → Copy

# Dán token vào đây (KHÔNG commit cell này lên GitHub)
# GH_TOKEN = 'ghp_xxxxxxxxxxxxxxxxxxxx'
# REPO_URL_AUTH = f'https://{GH_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
# !git clone {REPO_URL_AUTH} {REPO_DIR}

## 2. Mount Google Drive + link data

**Lần đầu:** Sẽ có popup hỏi quyền → click **"Connect to Google Drive"** → chọn tài khoản → **Allow**.

**Cấu trúc thư mục cần có trên Drive của bạn:**
```
MyDrive/
└── colab_sentiment/
    ├── data/
    │   ├── raw/
    │   │   ├── crawled_emotions.xlsx       ← upload file này
    │   │   ├── unlabeled_new_posts.json    ← upload file này
    │   │   └── UIT-VSMEC.csv               ← upload sau khi tải về
    │   └── processed/                      ← tự động tạo
    └── models/                             ← checkpoints lưu ở đây
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

In [ ]:
import os

# ── Cấu hình đường dẫn Drive ────────────────────────────────────────────────
DRIVE_ROOT      = '/content/drive/MyDrive/colab_sentiment'
DRIVE_DATA_RAW  = f'{DRIVE_ROOT}/data/raw'
DRIVE_DATA_PROC = f'{DRIVE_ROOT}/data/processed'
DRIVE_MODELS    = f'{DRIVE_ROOT}/models'
DRIVE_REPORTS   = f'{DRIVE_ROOT}/reports'

# Tạo thư mục trên Drive nếu chưa có
for d in [DRIVE_DATA_RAW, DRIVE_DATA_PROC, DRIVE_MODELS, DRIVE_REPORTS]:
    os.makedirs(d, exist_ok=True)
    print(f'  ✅ {d}')

# ── Symlink: trỏ data/ và models/ trong repo → Drive ────────────────────────
# Mục đích: code trong repo dùng path tương đối (data/, models/) như bình thường,
# nhưng thực ra đọc/ghi vào Drive → không mất sau khi Colab reset.

def _symlink(src, dst):
    """Tạo symlink dst → src, bỏ qua nếu đã tồn tại."""
    if os.path.islink(dst):
        os.remove(dst)
    elif os.path.isdir(dst) and not os.path.islink(dst):
        import shutil
        shutil.rmtree(dst)   # xoá thư mục placeholder trong repo
    os.symlink(src, dst)
    print(f'  linked: {dst} → {src}')

os.chdir(REPO_DIR)
_symlink(DRIVE_ROOT + '/data',    'data')
_symlink(DRIVE_MODELS,            'models')
_symlink(DRIVE_REPORTS,           'reports')

# Tạo .gitkeep để git không than phiền
for sub in ['data/raw', 'data/processed', 'data/external']:
    os.makedirs(sub, exist_ok=True)

print('\n✅ Symlinks created. Verifying...')
!ls data/raw/

In [ ]:
# Kiểm tra file data cần có
required_files = [
    'data/raw/crawled_emotions.xlsx',
    'data/raw/unlabeled_new_posts.json',
]
optional_files = [
    'data/raw/UIT-VSMEC.csv',
    'data/processed/cleaned_unlabeled_posts.csv',
    'data/processed/pseudo_labeled_apify.csv',
]

print('Required files:')
all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = '✅' if exists else '❌ MISSING — upload lên Drive trước'
    print(f'  {status}  {f}')
    if not exists:
        all_ok = False

print('\nOptional files (sẽ được tạo tự động):')
for f in optional_files:
    exists = os.path.exists(f)
    print(f'  {"✅" if exists else "⬜ chưa có"} {f}')

if not all_ok:
    print('\n⚠️  Upload file còn thiếu lên Google Drive trước khi tiếp tục.')
    print(f'   Đường dẫn Drive: {DRIVE_DATA_RAW}/')

## 3. Cài thư viện

Khoảng 3–5 phút lần đầu. Colab đã có sẵn torch/numpy nên nhanh hơn local.

In [ ]:
# Cache pip để không phải reinstall nếu session reconnect vào cùng runtime
import subprocess, sys

CACHE_FLAG = f'{DRIVE_ROOT}/.pip_installed'

if os.path.exists(CACHE_FLAG):
    print('✅ Pip cache found — installing from requirements.txt (fast path)...')
else:
    print('📦 First-time install — this takes ~3-5 minutes...')

!pip install -q -r requirements.txt 2>&1 | tail -5

# Đánh dấu đã install
open(CACHE_FLAG, 'w').write('ok')
print('\n✅ Dependencies installed.')

In [ ]:
# Verify các import quan trọng
import torch, transformers, pandas, numpy, sklearn, rtdl
print('torch      :', torch.__version__)
print('transformers:', transformers.__version__)
print('pandas     :', pandas.__version__)
print('sklearn    :', sklearn.__version__)
print('CUDA       :', torch.cuda.is_available())
print('\n✅ All imports OK')

## 4. Pseudo-label 990 Facebook posts

Dùng mDeBERTa zero-shot để tự động gán nhãn emotion cho unlabeled Apify posts.
- **Lần đầu**: tải model ~560MB (lưu vào HuggingFace cache trên Drive).
- **Lần sau**: load từ cache, nhanh hơn nhiều.

In [ ]:
import os

# Cache HuggingFace models vào Drive để không tải lại mỗi session
HF_CACHE = f'{DRIVE_ROOT}/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
print(f'✅ HuggingFace cache → {HF_CACHE}')

In [ ]:
PSEUDO_OUTPUT = 'data/processed/pseudo_labeled_apify.csv'

if os.path.exists(PSEUDO_OUTPUT):
    print(f'✅ Pseudo-labeled file already exists: {PSEUDO_OUTPUT}')
    print('   Bỏ qua bước này. Xoá file trên Drive nếu muốn chạy lại.')
else:
    print('🔄 Running pseudo-labeling (~10-20 min on GPU)...')
    !python -m scripts.pseudo_label_apify \
        --input  data/processed/cleaned_unlabeled_posts.csv \
        --output {PSEUDO_OUTPUT} \
        --model  MoritzLaurer/mDeBERTa-v3-base-mnli-xnli \
        --batch-size 32 \
        --threshold  0.35 \
        --device cuda

In [ ]:
import pandas as pd
if os.path.exists(PSEUDO_OUTPUT):
    pseudo = pd.read_csv(PSEUDO_OUTPUT)
    print(f'Pseudo-labeled posts: {len(pseudo):,}')
    print(f'Confident (≥0.35)   : {pseudo["pseudo_confident"].sum():,}')
    print('\nLabel distribution:')
    print(pseudo['label'].value_counts().to_string())
    print('\nConfidence stats:')
    print(pseudo['pseudo_confidence'].describe().round(3).to_string())

## 5. Chuẩn bị dataset — merge tất cả nguồn

In [ ]:
# Build command tuỳ theo file nào có sẵn
cmd = 'python -m scripts.prepare_data'
cmd += ' --crawled        data/raw/crawled_emotions.xlsx'
cmd += ' --output-dir     data/processed'
cmd += ' --seed           42'

if os.path.exists('data/raw/UIT-VSMEC.csv'):
    cmd += ' --uit-vsmec  data/raw/UIT-VSMEC.csv'
    print('✅ UIT-VSMEC detected — will be merged.')
else:
    print('⬜ UIT-VSMEC not found — skipping (upload data/raw/UIT-VSMEC.csv to add ~7000 samples).')

if os.path.exists('data/processed/pseudo_labeled_apify.csv'):
    cmd += ' --pseudo-labeled data/processed/pseudo_labeled_apify.csv'
    cmd += ' --confidence-threshold 0.35'
    print('✅ Pseudo-labeled Apify detected — will be merged.')
else:
    print('⬜ Pseudo-labeled not found — run Section 4 first.')

print(f'\n▶ Running: {cmd}\n')
!{cmd}

In [ ]:
import pandas as pd

for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'data/processed/{split}.parquet')
    print(f'{split:5s}: {len(df):>5,} rows | {len(df.columns)} cols | '
          f'label dist: {df["label"].value_counts().to_dict()}')

# Verify tabular columns are present
train = pd.read_parquet('data/processed/train.parquet')
tabular_cols = ['text_length','n_words','n_exclamation','n_question',
                'n_emoji_token','n_hashtag','n_latin_words',
                'likes','comments','shares','has_emoji','is_crawled']
missing = [c for c in tabular_cols if c not in train.columns]
if missing:
    print(f'\n⚠️  Missing tabular cols: {missing}')
else:
    print(f'\n✅ All {len(tabular_cols)} tabular columns present')

## 6. Training — Late Fusion XLM-R + FT-Transformer

Thời gian ước tính:
- T4 GPU (Colab free): ~1.5–2.5 giờ / 10 epochs với ~8000 mẫu
- A100 GPU (Colab Pro+): ~30–45 phút

In [ ]:
import yaml, os

# Load config và override output dir → Drive để không mất checkpoint
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Override output → Drive
cfg['training']['output_dir'] = f'{DRIVE_MODELS}/best_model'
cfg['training']['device']     = DEVICE

# Ghi config tạm
COLAB_CONFIG = '/tmp/config_colab.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

print('Training config:')
print(f"  output_dir  : {cfg['training']['output_dir']}")
print(f"  device      : {cfg['training']['device']}")
print(f"  epochs      : {cfg['training'].get('epochs', 10)}")
print(f"  batch_size  : {cfg['training'].get('batch_size', 32)}")
print(f"  learning_rate: {cfg['training'].get('learning_rate', 2e-5)}")

In [ ]:
# Chạy training — output sẽ stream trực tiếp
!python -m src.train --config {COLAB_CONFIG}

In [ ]:
# Verify checkpoint đã được lưu vào Drive
ckpt_dir = f'{DRIVE_MODELS}/best_model'
if os.path.exists(ckpt_dir):
    files = os.listdir(ckpt_dir)
    print(f'✅ Checkpoint saved to Drive: {ckpt_dir}')
    for f in files:
        size = os.path.getsize(f'{ckpt_dir}/{f}') / 1e6
        print(f'   {f}: {size:.1f} MB')
else:
    print('❌ Checkpoint not found — check training logs above')

## 7. Ablation Study (3 experiments)

**Quan trọng nhất cho báo cáo** — chứng minh đóng góp từng thành phần.
Thời gian: ~3× thời gian training = 4–8 giờ.

**Chạy section này sau khi training xong** hoặc trên session riêng.

In [ ]:
ABLATION_OUTPUT = f'{DRIVE_MODELS}/ablation'

!python -m scripts.run_ablation \
    --raw        data/raw/crawled_emotions.xlsx \
    --output-dir {ABLATION_OUTPUT} \
    --epochs     4 \
    --batch-size 32 \
    --device     {DEVICE}

In [ ]:
# Hiện bảng kết quả ablation
import pandas as pd

results_path = 'reports/ablation_results.csv'
if os.path.exists(results_path):
    df = pd.read_csv(results_path, index_col=0)
    display_cols = ['use_normalizer','use_tabular','f1_macro',
                    'precision_macro','recall_macro','accuracy']
    print('\n===== ABLATION RESULTS =====')
    print(df[display_cols].to_string(float_format=lambda v: f'{v:.4f}'))

    # Copy sang Drive
    import shutil
    shutil.copy(results_path, f'{DRIVE_REPORTS}/ablation_results.csv')
    print(f'\n✅ Results copied → Drive')

## 8. Evaluate trên Test Set + Save kết quả

In [ ]:
CHECKPOINT = f'{DRIVE_MODELS}/best_model'

if not os.path.exists(CHECKPOINT):
    print('❌ Checkpoint not found — run Section 6 first.')
else:
    !python -m src.evaluate \
        --checkpoint {CHECKPOINT} \
        --data       data/processed/test.parquet \
        --output-dir reports/

In [ ]:
# Copy toàn bộ reports → Drive để không mất sau session
import shutil, glob

for f in glob.glob('reports/**/*', recursive=True):
    if os.path.isfile(f):
        dest = f'{DRIVE_ROOT}/{f}'
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy2(f, dest)

print('✅ Reports copied → Google Drive')
!ls {DRIVE_REPORTS}/

## Tips & Troubleshooting

| Vấn đề | Giải pháp |
|---|---|
| Session bị disconnect / reset | Chạy lại từ Section 1 — data/models vẫn còn trên Drive |
| `CUDA out of memory` | Giảm `batch_size` trong config.yaml xuống 16 hoặc 8 |
| HuggingFace download chậm | Đã cache trên Drive — lần 2 sẽ nhanh hơn |
| `ModuleNotFoundError` | Chạy lại Section 3 (pip install) |
| Colab disconnect sau 12h | Dùng Colab Pro hoặc tắt `early_stopping` để training nhanh hơn |
| Drive symlink lỗi | Chạy lại Section 2 (mount + symlink) |